# Create embeddings of documents

### Quickstart Example

In [27]:
import os
from dotenv import load_dotenv
from collections import defaultdict
from openai import OpenAI
import pickle
import pandas as pd

In [5]:
load_dotenv()
open_api_key = os.getenv("OPENAI_API_KEY")

In [21]:
client = OpenAI(api_key=open_api_key)

# ToDo: add some sort of loop through all videos
video_id = '2Ds1m5gflCI'

with open(f'data/documents/{video_id}.txt', 'r', encoding='utf-8') as file:
    text_content = file.read()

from semantic_text_splitter import TextSplitter
from tokenizers import Tokenizer

# ToDo: find max length
max_tokens = 200
tokenizer = Tokenizer.from_pretrained("bert-base-uncased")
splitter = TextSplitter.from_huggingface_tokenizer(tokenizer, max_tokens)

chunks = splitter.chunks(text_content)

In [22]:
embeddings_dict = defaultdict(lambda: defaultdict(list))

for chunk in chunks:
    # Create chunks directory if it doesn't exist
    os.makedirs('data/chunks', exist_ok=True)
    
    # Write chunk to file with video ID and chunk number
    chunk_filename = f'data/chunks/{video_id}_chunk{chunks.index(chunk)}.txt'
    with open(chunk_filename, 'w', encoding='utf-8') as f:
        f.write(chunk)
    response = client.embeddings.create(
        input=chunk,
        model="text-embedding-3-large"
    )

    # Store embeddings in a dictionary
    chunk_id = f'chunk_{chunks.index(chunk)}'
    embedding = response.data[0].embedding
    embeddings_dict[video_id][chunk_id] = embedding



Potential design options:
1. create a list of dictionaries and store the data in a dataframe using columns: 'embedding', 'rank', 'chunk_id', 'video_id'
2. redis
3. vector db
4. mongo collections
5. pickle

In [13]:
# Save the defaultdict to a file
def save_defaultdict(data, filename):
    with open(filename, 'wb') as file:
        pickle.dump(dict(data), file)

In [14]:
# Load the defaultdict from the file
def load_defaultdict(filename):
    with open(filename, 'rb') as file:
        loaded_dict = pickle.load(file)
        restored_dict = defaultdict(lambda: defaultdict(list), loaded_dict)
    return restored_dict

In [24]:
save_defaultdict(embeddings_dict, 'data/embeddings_dict.pkl')
loaded_dict = load_defaultdict('data/embeddings_dict.pkl')

In [25]:
loaded_dict == embeddings_dict

True

In [18]:
loaded_dict['2Ds1m5gflCI']

defaultdict(list,
            {'chunk_0': [0.02152094431221485,
              -0.02277633175253868,
              -0.01966775208711624,
              -0.005432543810456991,
              -0.005634302739053965,
              -0.01880093663930893,
              0.009041785262525082,
              0.021386438980698586,
              0.02618381567299366,
              0.01884577050805092,
              -0.006553426384925842,
              0.01942862942814827,
              0.00941541325300932,
              0.01724664494395256,
              -0.002983415499329567,
              -0.012083113193511963,
              -0.04558255523443222,
              0.022073913365602493,
              -0.04244408383965492,
              -0.007935848087072372,
              0.011335858143866062,
              -0.011866409331560135,
              -0.030906466767191887,
              -0.017724888399243355,
              -0.010872560553252697,
              -0.009564864449203014,
              0.02699085138738

ToDo:
- Add documentation
- Move funtions to utils
- Adjust max length of chunks
- Add chunks to Mongo
- Make Query embedding function
- Rank Query to embedding with chunks
    - Will need to engineer a data object to handle this 
        - Add a secondary "key" object?
- Generate response using top chunks